## 2.1 理论计算题

给定字符序列 `"ababc"`，词汇表为 {`'a'`, `'b'`, `'c'`}，采用一阶马尔可夫模型，使用拉普拉斯平滑（加1平滑）估计条件概率。

### 转移计数（未平滑）

序列中相邻字符对（转移）为：
- `a -> b`
- `b -> a`
- `a -> b`
- `b -> c`

计数矩阵（行 → 列）：

| 行\列 | a | b | c |
|-------|---|---|---|
| a     | 0 | 2 | 0 |
| b     | 1 | 0 | 1 |
| c     | 0 | 0 | 0 |

### 拉普拉斯平滑（加1）

平滑后计数：`count'(x, y) = count(x, y) + 1`

| 行\列 | a | b | c | 行总和 |
|-------|---|---|---|--------|
| a     | 1 | 3 | 1 | 5      |
| b     | 2 | 1 | 2 | 5      |
| c     | 1 | 1 | 1 | 3      |

条件概率公式：  
`p(y | x) = count'(x, y) / sum_y count'(x, y)`

---

### 1. p(a | b)

`p(a | b) = count'(b, a) / (count'(b, a) + count'(b, b) + count'(b, c))`
`= 2 / (2 + 1 + 2) = 2 / 5 = 0.4`

### 2. p(c | b)

`p(c | b) = count'(b, c) / (count'(b, a) + count'(b, b) + count'(b, c))`
`= 2 / (2 + 1 + 2) = 2 / 5 = 0.4`

---

**答案：**
1. `p(a | b) = 2/5 = 0.4`
2. `p(c | b) = 2/5 = 0.4`

In [2]:
import re
from collections import Counter

def preprocess_text(text, n):
    # 1. 转换为小写，去除标点符号（保留字母和空格）
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 2. 按空格分词
    words = text.split()
    
    # 3. 构建词汇表（按出现频率排序，分配整数ID，从0开始）
    word_counts = Counter(words)
    sorted_words = sorted(word_counts.keys(), key=lambda w: (-word_counts[w], w))
    vocab = {word: idx for idx, word in enumerate(sorted_words)}
    
    # 4. 滑动窗口生成特征序列和标签
    features = []
    labels = []
    for i in range(len(words) - n):
        features.append(words[i:i+n])
        labels.append(words[i+n])
    
    # 返回词汇表字典和（特征列表，标签列表）
    return vocab, (features, labels)


# 测试案例
if __name__ == "__main__":
    # 示例测试
    test_text = "The time machine"
    n = 2
    vocab, (features, labels) = preprocess_text(test_text, n)
    
    print("词汇表:", vocab)
    print("特征序列:", features)
    print("标签序列:", labels)
    
    # 额外测试
    print("\n--- 额外测试 ---")
    test_text2 = "Hello world! This is a test. Test, test."
    n = 3
    vocab2, (features2, labels2) = preprocess_text(test_text2, n)
    
    print("词汇表:", vocab2)
    print("特征序列:", features2)
    print("标签序列:", labels2)

词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征序列: [['the', 'time']]
标签序列: ['machine']

--- 额外测试 ---
词汇表: {'test': 0, 'a': 1, 'hello': 2, 'is': 3, 'this': 4, 'world': 5}
特征序列: [['hello', 'world', 'this'], ['world', 'this', 'is'], ['this', 'is', 'a'], ['is', 'a', 'test'], ['a', 'test', 'test']]
标签序列: ['is', 'a', 'test', 'test', 'test']


## 3.1 理论计算题

考虑一个线性RNN（无偏置），定义为：
- \(h_t = W_{hh}h_{t-1} + W_{hx}x_t\)
- \(o_t = W_{oh}h_t\)
- 损失函数：\(L = \frac{1}{2}\sum_{t=1}^{T}(o_t - y_t)^2\)

### 推导损失对权重 \(W_{hh}\) 的梯度表达式

#### 步骤1：定义局部误差
令 \(\delta_t = \frac{\partial L}{\partial h_t}\)，则：
\[\delta_t = \frac{\partial L}{\partial o_t} \cdot \frac{\partial o_t}{\partial h_t} = (o_t - y_t) \cdot W_{oh}^T\]

#### 步骤2：通过时间反向传播
对于时间步 \(t\)，隐藏状态 \(h_t\) 会影响：
- 当前时间步的输出 \(o_t\)
- 未来时间步的隐藏状态 \(h_{t+1}, h_{t+2}, ..., h_T\)

因此：
\[\delta_t = (o_t - y_t)W_{oh}^T + W_{hh}^T \delta_{t+1}\]

其中 \(\delta_{T+1} = 0\)（边界条件）

#### 步骤3：展开到所有时间步
将 \(\delta_t\) 展开：
\[\delta_t = \sum_{k=t}^{T} (W_{hh}^T)^{k-t} (o_k - y_k)W_{oh}^T\]

#### 步骤4：计算对 \(W_{hh}\) 的梯度
\[\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \frac{\partial L}{\partial h_t} \cdot \frac{\partial h_t}{\partial W_{hh}} = \sum_{t=1}^{T} \delta_t h_{t-1}^T\]

代入 \(\delta_t\) 的展开式：
\[\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \sum_{k=t}^{T} (W_{hh}^T)^{k-t} (o_k - y_k)W_{oh}^T h_{t-1}^T\]

### 梯度消失或爆炸的条件

梯度消失或爆炸取决于雅可比矩阵 \(W_{hh}^T\) 的特征值：

- **梯度消失**：当 \(W_{hh}\) 的最大特征值 \(\lambda_{\max} < 1\) 时，随着 \(k-t\) 增大，\((W_{hh}^T)^{k-t}\) 指数衰减，导致梯度趋近于0。

- **梯度爆炸**：当 \(W_{hh}\) 的最大特征值 \(\lambda_{\max} > 1\) 时，随着 \(k-t\) 增大，\((W_{hh}^T)^{k-t}\) 指数增长，导致梯度趋近于无穷大。

- **临界情况**：当 \(\lambda_{\max} = 1\) 时，梯度既不消失也不爆炸（但可能出现其他稳定性问题）。

**结论**：梯度消失或爆炸的条件为 \(|\lambda_{\max}(W_{hh})| \neq 1\)，其中 \(\lambda_{\max}\) 为权重矩阵 \(W_{hh}\) 的最大特征值。

In [3]:
import numpy as np

def rnn_cell_forward(x_t, h_prev, W_hh, W_xh, b_h):
    """
    RNN单元前向传播
    
    参数:
        x_t: 当前输入，形状 (batch_size, input_size)
        h_prev: 上一隐藏状态，形状 (batch_size, hidden_size)
        W_hh: 隐藏状态权重，形状 (hidden_size, hidden_size)
        W_xh: 输入权重，形状 (input_size, hidden_size)
        b_h: 偏置，形状 (hidden_size,)
    
    返回:
        h_t: 当前隐藏状态，形状 (batch_size, hidden_size)
        cache: 缓存中间变量用于反向传播
    """
    # 计算线性变换
    h_linear = np.dot(x_t, W_xh) + np.dot(h_prev, W_hh) + b_h
    
    # tanh激活
    h_t = np.tanh(h_linear)
    
    # 缓存中间变量
    cache = (x_t, h_prev, h_linear, h_t, W_hh, W_xh, b_h)
    
    return h_t, cache


def rnn_cell_backward(dh_next, cache):
    """
    RNN单元单步反向传播
    
    参数:
        dh_next: 损失对当前隐藏状态h_t的梯度，形状 (batch_size, hidden_size)
        cache: 前向传播缓存的中间变量
    
    返回:
        dx_t: 损失对输入x_t的梯度，形状 (batch_size, input_size)
        dh_prev: 损失对上一隐藏状态h_prev的梯度，形状 (batch_size, hidden_size)
        dW_hh: 损失对W_hh的梯度，形状 (hidden_size, hidden_size)
        dW_xh: 损失对W_xh的梯度，形状 (input_size, hidden_size)
        db_h: 损失对b_h的梯度，形状 (hidden_size,)
    """
    # 解包缓存
    x_t, h_prev, h_linear, h_t, W_hh, W_xh, b_h = cache
    
    # tanh激活函数的梯度: dtanh/dz = 1 - tanh(z)^2
    dh_linear = dh_next * (1 - h_t ** 2)  # 形状 (batch_size, hidden_size)
    
    # 计算各参数的梯度
    # 1. 对b_h的梯度
    db_h = np.sum(dh_linear, axis=0)  # 形状 (hidden_size,)
    
    # 2. 对W_hh的梯度
    dW_hh = np.dot(h_prev.T, dh_linear)  # 形状 (hidden_size, hidden_size)
    
    # 3. 对W_xh的梯度
    dW_xh = np.dot(x_t.T, dh_linear)  # 形状 (input_size, hidden_size)
    
    # 4. 对h_prev的梯度
    dh_prev = np.dot(dh_linear, W_hh.T)  # 形状 (batch_size, hidden_size)
    
    # 5. 对x_t的梯度
    dx_t = np.dot(dh_linear, W_xh.T)  # 形状 (batch_size, input_size)
    
    return dx_t, dh_prev, dW_hh, dW_xh, db_h


# 测试示例
if __name__ == "__main__":
    # 设置参数
    batch_size = 2
    input_size = 3
    hidden_size = 4
    
    # 初始化数据
    np.random.seed(42)
    x_t = np.random.randn(batch_size, input_size)
    h_prev = np.random.randn(batch_size, hidden_size)
    W_hh = np.random.randn(hidden_size, hidden_size)
    W_xh = np.random.randn(input_size, hidden_size)
    b_h = np.random.randn(hidden_size)
    
    # 前向传播
    h_t, cache = rnn_cell_forward(x_t, h_prev, W_hh, W_xh, b_h)
    print("前向传播结果:")
    print(f"h_t 形状: {h_t.shape}")
    print(f"h_t 前2行:\n{h_t[:2]}")
    
    # 反向传播
    dh_next = np.random.randn(batch_size, hidden_size)
    dx_t, dh_prev, dW_hh, dW_xh, db_h = rnn_cell_backward(dh_next, cache)
    
    print("\n反向传播结果:")
    print(f"dx_t 形状: {dx_t.shape}")
    print(f"dh_prev 形状: {dh_prev.shape}")
    print(f"dW_hh 形状: {dW_hh.shape}")
    print(f"dW_xh 形状: {dW_xh.shape}")
    print(f"db_h 形状: {db_h.shape}")
    
    # 梯度检查（简单数值验证）
    print("\n梯度检查:")
    epsilon = 1e-5
    W_hh_plus = W_hh.copy()
    W_hh_plus[0, 0] += epsilon
    h_t_plus, _ = rnn_cell_forward(x_t, h_prev, W_hh_plus, W_xh, b_h)
    h_t_minus, _ = rnn_cell_forward(x_t, h_prev, W_hh - epsilon * np.eye(hidden_size)[0:1, 0:1], W_xh, b_h)
    numerical_grad = np.sum((h_t_plus - h_t_minus) * dh_next) / (2 * epsilon)
    print(f"数值梯度 (W_hh[0,0]): {numerical_grad:.6f}")
    print(f"解析梯度 (W_hh[0,0]): {dW_hh[0,0]:.6f}")
    print(f"差异: {abs(numerical_grad - dW_hh[0,0]):.6f}")

前向传播结果:
h_t 形状: (2, 4)
h_t 前2行:
[[-0.99996421 -0.18180438 -0.91872724 -0.6355189 ]
 [ 0.98731536  0.98933976 -0.72430849 -0.87813856]]

反向传播结果:
dx_t 形状: (2, 3)
dh_prev 形状: (2, 4)
dW_hh 形状: (4, 4)
dW_xh 形状: (3, 4)
db_h 形状: (4,)

梯度检查:
数值梯度 (W_hh[0,0]): 0.264424
解析梯度 (W_hh[0,0]): -0.003838
差异: 0.268262


## 4.1 理论计算题

假设一个深度双向RNN，有L层，每层隐藏单元数为H，输入维度为D，输出维度为O（仅考虑最后输出层）。计算该模型的参数总数（包括所有全连接层的权重和偏置），忽略嵌入层和输出层之前的投影。

### 参数计算

#### 1. 第一层（输入层 → 第一层隐藏层）

**前向RNN方向：**
- 权重 $W_{xh}^{(1,f)}$：$D \times H$，偏置 $b_h^{(1,f)}$：$H$
- 隐藏状态权重 $W_{hh}^{(1,f)}$：$H \times H$，偏置 $b_{hh}^{(1,f)}$：$H$

**后向RNN方向：**
- 权重 $W_{xh}^{(1,b)}$：$D \times H$，偏置 $b_h^{(1,b)}$：$H$
- 隐藏状态权重 $W_{hh}^{(1,b)}$：$H \times H$，偏置 $b_{hh}^{(1,b)}$：$H$

第一层参数：
- 前向：$D \times H + H + H \times H + H = H(D + H + 2)$
- 后向：$D \times H + H + H \times H + H = H(D + H + 2)$
- 总计：$2H(D + H + 2)$

#### 2. 中间层（第l层，$2 \leq l \leq L$）

**前向RNN方向：**
- 输入来自上一层的拼接隐藏状态（前向+后向），维度为 $2H$
- 输入权重 $W_{xh}^{(l,f)}$：$2H \times H$，偏置 $b_h^{(l,f)}$：$H$
- 隐藏状态权重 $W_{hh}^{(l,f)}$：$H \times H$，偏置 $b_{hh}^{(l,f)}$：$H$

**后向RNN方向：**
- 输入权重 $W_{xh}^{(l,b)}$：$2H \times H$，偏置 $b_h^{(l,b)}$：$H$
- 隐藏状态权重 $W_{hh}^{(l,b)}$：$H \times H$，偏置 $b_{hh}^{(l,b)}$：$H$

每层参数：
- 前向：$2H \times H + H + H \times H + H = H(3H + 2)$
- 后向：$2H \times H + H + H \times H + H = H(3H + 2)$
- 总计：$2H(3H + 2)$

#### 3. 最后一层到输出层

**输出层：**
- 最后一层的拼接隐藏状态（前向+后向），维度为 $2H$
- 输出权重 $W_{oh}$：$2H \times O$，偏置 $b_o$：$O$
- 参数：$2H \times O + O = O(2H + 1)$

### 总参数

总参数 = 第一层参数 + 中间层参数 + 输出层参数

$$ \text{Total} = 2H(D + H + 2) + (L-1) \cdot 2H(3H + 2) + O(2H + 1) $$

**简化表达式：**

$$ \text{Total} = 2H(D + H + 2) + 2H(L-1)(3H + 2) + O(2H + 1) $$

**展开形式：**

$$ \text{Total} = 2HD + 2H^2 + 4H + 6H^2(L-1) + 4H(L-1) + 2HO + O $$

$$ \text{Total} = 2HD + 2H^2 + 4H + 6H^2L - 6H^2 + 4HL - 4H + 2HO + O $$

$$ \text{Total} = 2HD + 6H^2L - 4H^2 + 4HL + 2HO + O $$

### 最终表达式

$$ \boxed{\text{Total} = 2H(D + H + 2) + 2H(L-1)(3H + 2) + O(2H + 1)} $$

或等价地：

$$ \boxed{\text{Total} = 2HD + 6H^2L - 4H^2 + 4HL + 2HO + O} $$

In [4]:
import torch
import torch.nn as nn

class BidirectionalRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        """
        双向RNN编码器
        
        参数:
            input_dim: 输入维度
            hidden_dim: 隐藏状态维度
            num_layers: RNN层数
        """
        super(BidirectionalRNNEncoder, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # 使用PyTorch的RNN实现双向
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=False  # 保持形状 (seq_len, batch, input_dim)
        )
    
    def forward(self, X):
        """
        前向传播
        
        参数:
            X: 输入序列，形状 (seq_len, batch, input_dim)
        
        返回:
            outputs: 每个时间步的拼接隐藏状态，形状 (seq_len, batch, 2*hidden_dim)
            final_state: 最终时间步的拼接隐藏状态，形状 (batch, 2*hidden_dim)
        """
        # RNN前向传播
        # outputs: (seq_len, batch, num_directions * hidden_dim)
        # h_n: (num_layers * num_directions, batch, hidden_dim)
        outputs, h_n = self.rnn(X)
        
        # 获取最后时间步的前向和后向隐藏状态
        # h_n形状: (num_layers * 2, batch, hidden_dim)
        # 前向最后状态: h_n[-2, :, :]
        # 后向最后状态: h_n[-1, :, :]
        forward_last = h_n[-2, :, :]  # (batch, hidden_dim)
        backward_last = h_n[-1, :, :]  # (batch, hidden_dim)
        
        # 拼接最终隐藏状态
        final_state = torch.cat([forward_last, backward_last], dim=-1)  # (batch, 2*hidden_dim)
        
        return outputs, final_state


# 手动实现的双向RNN编码器（不使用torch.nn.RNN）
class BidirectionalRNNEncoderManual(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        """
        手动实现的双向RNN编码器
        
        参数:
            input_dim: 输入维度
            hidden_dim: 隐藏状态维度
            num_layers: RNN层数
        """
        super(BidirectionalRNNEncoderManual, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # 前向RNN层
        self.rnn_forward = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=False,
            batch_first=False
        )
        
        # 后向RNN层（处理反转序列）
        self.rnn_backward = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=False,
            batch_first=False
        )
    
    def forward(self, X):
        """
        前向传播
        
        参数:
            X: 输入序列，形状 (seq_len, batch, input_dim)
        
        返回:
            outputs: 每个时间步的拼接隐藏状态，形状 (seq_len, batch, 2*hidden_dim)
            final_state: 最终时间步的拼接隐藏状态，形状 (batch, 2*hidden_dim)
        """
        seq_len, batch, _ = X.shape
        
        # 前向传播
        forward_outputs, _ = self.rnn_forward(X)  # (seq_len, batch, hidden_dim)
        
        # 后向传播：反转序列
        X_reversed = torch.flip(X, dims=[0])  # (seq_len, batch, input_dim)
        backward_outputs_reversed, _ = self.rnn_backward(X_reversed)  # (seq_len, batch, hidden_dim)
        backward_outputs = torch.flip(backward_outputs_reversed, dims=[0])  # (seq_len, batch, hidden_dim)
        
        # 拼接前向和后向输出
        outputs = torch.cat([forward_outputs, backward_outputs], dim=-1)  # (seq_len, batch, 2*hidden_dim)
        
        # 获取最终时间步的拼接隐藏状态
        forward_last = forward_outputs[-1, :, :]  # (batch, hidden_dim)
        backward_last = backward_outputs[-1, :, :]  # (batch, hidden_dim)
        final_state = torch.cat([forward_last, backward_last], dim=-1)  # (batch, 2*hidden_dim)
        
        return outputs, final_state


# 测试代码
if __name__ == "__main__":
    # 设置参数
    seq_len = 5
    batch = 3
    input_dim = 4
    hidden_dim = 6
    num_layers = 2
    
    # 创建随机输入
    X = torch.randn(seq_len, batch, input_dim)
    
    print("输入形状:", X.shape)
    print("-" * 50)
    
    # 测试PyTorch实现
    encoder1 = BidirectionalRNNEncoder(input_dim, hidden_dim, num_layers)
    outputs1, final_state1 = encoder1(X)
    
    print("PyTorch RNN实现:")
    print(f"输出形状: {outputs1.shape}")  # 期望: (seq_len, batch, 2*hidden_dim)
    print(f"最终状态形状: {final_state1.shape}")  # 期望: (batch, 2*hidden_dim)
    print()
    
    # 测试手动实现
    encoder2 = BidirectionalRNNEncoderManual(input_dim, hidden_dim, num_layers)
    outputs2, final_state2 = encoder2(X)
    
    print("手动实现:")
    print(f"输出形状: {outputs2.shape}")  # 期望: (seq_len, batch, 2*hidden_dim)
    print(f"最终状态形状: {final_state2.shape}")  # 期望: (batch, 2*hidden_dim)
    print()
    
    # 验证形状
    assert outputs1.shape == (seq_len, batch, 2*hidden_dim), "输出形状错误"
    assert final_state1.shape == (batch, 2*hidden_dim), "最终状态形状错误"
    assert outputs2.shape == (seq_len, batch, 2*hidden_dim), "手动实现输出形状错误"
    assert final_state2.shape == (batch, 2*hidden_dim), "手动实现最终状态形状错误"
    
    print("✓ 所有形状验证通过")
    
    # 验证输出内容（选做）
    print("\n示例输出值（PyTorch实现）:")
    print(f"第一个时间步的前两个batch的输出（前5个维度）:\n{outputs1[0, :2, :5]}")
    print(f"\n最终状态（前两个batch）:\n{final_state1[:2, :]}")

输入形状: torch.Size([5, 3, 4])
--------------------------------------------------
PyTorch RNN实现:
输出形状: torch.Size([5, 3, 12])
最终状态形状: torch.Size([3, 12])

手动实现:
输出形状: torch.Size([5, 3, 12])
最终状态形状: torch.Size([3, 12])

✓ 所有形状验证通过

示例输出值（PyTorch实现）:
第一个时间步的前两个batch的输出（前5个维度）:
tensor([[ 0.2782, -0.3487,  0.0192,  0.7122,  0.6194],
        [ 0.1562, -0.5437, -0.0719,  0.5356,  0.6216]],
       grad_fn=<SliceBackward0>)

最终状态（前两个batch）:
tensor([[ 0.0789, -0.5380, -0.3647,  0.1453,  0.6646, -0.3065, -0.6784,  0.0675,
         -0.4585,  0.2873,  0.3210, -0.0449],
        [-0.1143,  0.6740, -0.5053, -0.1596,  0.7199, -0.4891, -0.5428, -0.2453,
         -0.7719,  0.4840,  0.6471,  0.0099]], grad_fn=<SliceBackward0>)


## 5.1 理论计算题

在Skip-gram模型中，给定中心词 $w_c$ 和上下文词 $w_o$，使用负采样（采样 $K$ 个负样本）。

### 损失函数推导

#### 1. 原始Skip-gram目标

对于给定的中心词 $w_c$ 和上下文词 $w_o$，原始目标是最大化：

$$ P(w_o | w_c) = \frac{\exp(\mathbf{u}_o^T \mathbf{v}_c)}{\sum_{w' \in V} \exp(\mathbf{u}_{w'}^T \mathbf{v}_c)} $$

对应的负对数似然损失为：

$$ L = -\log P(w_o | w_c) = -\mathbf{u}_o^T \mathbf{v}_c + \log \sum_{w' \in V} \exp(\mathbf{u}_{w'}^T \mathbf{v}_c) $$

#### 2. 负采样近似

负采样将多分类问题转化为二分类问题。对于每个正样本 $(w_c, w_o)$，我们采样 $K$ 个负样本 $(w_c, w_n)$。

目标函数变为最大化：

$$ \log \sigma(\mathbf{u}_o^T \mathbf{v}_c) + \sum_{k=1}^{K} \log \sigma(-\mathbf{u}_{n_k}^T \mathbf{v}_c) $$

其中 $\sigma(x) = \frac{1}{1 + \exp(-x)}$ 是sigmoid函数。

#### 3. 负对数似然损失

对应的损失函数（负对数似然）：

$$ L = -\log \sigma(\mathbf{u}_o^T \mathbf{v}_c) - \sum_{k=1}^{K} \log \sigma(-\mathbf{u}_{n_k}^T \mathbf{v}_c) $$

展开sigmoid函数：

$$ L = -\log \frac{1}{1 + \exp(-\mathbf{u}_o^T \mathbf{v}_c)} - \sum_{k=1}^{K} \log \frac{1}{1 + \exp(\mathbf{u}_{n_k}^T \mathbf{v}_c)} $$

$$ L = \log(1 + \exp(-\mathbf{u}_o^T \mathbf{v}_c)) + \sum_{k=1}^{K} \log(1 + \exp(\mathbf{u}_{n_k}^T \mathbf{v}_c)) $$

### 完整目标函数

$$ \boxed{L = -\log \sigma(\mathbf{u}_o^T \mathbf{v}_c) - \sum_{k=1}^{K} \log \sigma(-\mathbf{u}_{n_k}^T \mathbf{v}_c)} $$

或等价地：

$$ \boxed{L = \log(1 + \exp(-\mathbf{u}_o^T \mathbf{v}_c)) + \sum_{k=1}^{K} \log(1 + \exp(\mathbf{u}_{n_k}^T \mathbf{v}_c))} $$

其中：
- $\mathbf{v}_c$：中心词 $w_c$ 的输入词向量
- $\mathbf{u}_o$：上下文词 $w_o$ 的输出词向量
- $\mathbf{u}_{n_k}$：第 $k$ 个负样本词 $n_k$ 的输出词向量
- $K$：负样本数量
- $\sigma(x) = 1/(1 + e^{-x})$：sigmoid函数

### 负样本采样方法

从噪声分布 $P_n(w)$ 中采样负样本：

$$ P_n(w) = \frac{\text{freq}(w)^{3/4}}{\sum_{w' \in V} \text{freq}(w')^{3/4}} $$

其中 $\text{freq}(w)$ 是词 $w$ 在语料库中的频率。

**采样过程：**
1. 计算每个词的权重 $\text{freq}(w)^{3/4}$
2. 归一化得到概率分布 $P_n(w)$
3. 从该分布中独立采样 $K$ 个词作为负样本（不放回或放回采样）

常见的噪声分布还有均匀分布 $P_n(w) = 1/V$ 或基于词频的分布。

### 最终完整表达式

对于给定中心词 $w_c$、上下文词 $w_o$ 和负样本集合 $\{n_1, n_2, ..., n_K\}$：

$$ \boxed{L(w_c, w_o, \{n_k\}_{k=1}^K) = -\log \sigma(\mathbf{u}_o^T \mathbf{v}_c) - \sum_{k=1}^{K} \log \sigma(-\mathbf{u}_{n_k}^T \mathbf{v}_c)} $$

其中负样本 $n_k \sim P_n(w)$ 从噪声分布中采样。

In [5]:
import torch
import torch.nn.functional as F
import numpy as np

def cbow_forward(context_indices, target_indices, W, W_out):
    """
    CBOW模型的前向传播和损失计算（完整softmax）
    
    参数:
        context_indices: 上下文词索引列表，形状 (batch_size, context_size)
        target_indices: 目标中心词索引，形状 (batch_size,)
        W: 输入权重矩阵，形状 (V, d)
        W_out: 输出权重矩阵，形状 (d, V)
    
    返回:
        loss: 交叉熵损失值（标量）
    """
    batch_size, context_size = context_indices.shape
    V, d = W.shape
    
    # 1. 获取上下文词的嵌入向量
    # context_indices: (batch_size, context_size)
    # 对每个样本的每个上下文词，从W中获取对应的嵌入向量
    context_embeddings = W[context_indices]  # (batch_size, context_size, d)
    
    # 2. 计算平均上下文向量作为隐藏层
    # 对context_size维度求平均
    hidden = torch.mean(context_embeddings, dim=1)  # (batch_size, d)
    
    # 3. 计算输出概率分布
    # hidden: (batch_size, d) -> W_out: (d, V)
    scores = torch.matmul(hidden, W_out)  # (batch_size, V)
    
    # 使用softmax得到概率分布
    probs = F.softmax(scores, dim=1)  # (batch_size, V)
    
    # 4. 计算交叉熵损失
    # target_indices: (batch_size,)
    # 使用负对数似然损失（等价于交叉熵）
    loss = F.cross_entropy(scores, target_indices)  # 标量
    
    return loss, hidden, probs


# 带完整前向传播的CBOW（返回更多中间结果）
class CBOWModel:
    def __init__(self, vocab_size, embedding_dim):
        """
        初始化CBOW模型
        
        参数:
            vocab_size: 词汇表大小 V
            embedding_dim: 嵌入维度 d
        """
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        
        # 初始化权重矩阵
        # W: (V, d) - 输入权重
        # W_out: (d, V) - 输出权重
        self.W = torch.randn(vocab_size, embedding_dim) * 0.01
        self.W_out = torch.randn(embedding_dim, vocab_size) * 0.01
    
    def forward(self, context_indices, target_indices):
        """
        前向传播
        
        参数:
            context_indices: (batch_size, context_size)
            target_indices: (batch_size,)
        
        返回:
            loss: 损失值
            hidden: 隐藏层向量 (batch_size, d)
            probs: 输出概率分布 (batch_size, V)
        """
        batch_size, context_size = context_indices.shape
        
        # 获取上下文词嵌入
        context_embeddings = self.W[context_indices]  # (batch_size, context_size, d)
        
        # 平均得到隐藏层
        hidden = torch.mean(context_embeddings, dim=1)  # (batch_size, d)
        
        # 计算得分
        scores = torch.matmul(hidden, self.W_out)  # (batch_size, V)
        
        # 计算损失
        loss = F.cross_entropy(scores, target_indices)
        
        # 计算概率分布（用于输出）
        probs = F.softmax(scores, dim=1)
        
        return loss, hidden, probs


# 测试代码
if __name__ == "__main__":
    # 设置参数
    V = 10  # 词汇表大小
    d = 4   # 嵌入维度
    batch_size = 3
    context_size = 2
    
    # 创建随机数据
    torch.manual_seed(42)
    
    # 随机生成上下文词索引和目标词索引
    context_indices = torch.randint(0, V, (batch_size, context_size))
    target_indices = torch.randint(0, V, (batch_size,))
    
    # 随机初始化权重矩阵
    W = torch.randn(V, d) * 0.01
    W_out = torch.randn(d, V) * 0.01
    
    print("测试数据:")
    print(f"上下文索引 (batch_size={batch_size}, context_size={context_size}):")
    print(context_indices)
    print(f"\n目标索引 (batch_size={batch_size}):")
    print(target_indices)
    print(f"\n输入权重 W 形状: {W.shape}")
    print(f"输出权重 W_out 形状: {W_out.shape}")
    print("-" * 50)
    
    # 测试函数
    loss, hidden, probs = cbow_forward(context_indices, target_indices, W, W_out)
    
    print("CBOW前向传播结果:")
    print(f"损失值: {loss.item():.4f}")
    print(f"隐藏层形状: {hidden.shape}")  # (batch_size, d)
    print(f"隐藏层:\n{hidden}")
    print(f"\n输出概率分布形状: {probs.shape}")  # (batch_size, V)
    print(f"输出概率分布 (前3个词):\n{probs[:, :3]}")
    
    # 验证概率和为1
    print(f"\n验证: 每行概率和为 {probs.sum(dim=1)}")
    
    # 使用CBOW类
    print("\n" + "=" * 50)
    print("使用CBOW类测试:")
    model = CBOWModel(V, d)
    loss2, hidden2, probs2 = model.forward(context_indices, target_indices)
    print(f"损失值: {loss2.item():.4f}")
    print(f"隐藏层形状: {hidden2.shape}")
    print(f"概率分布形状: {probs2.shape}")
    
    # 梯度验证（可选）
    print("\n" + "=" * 50)
    print("梯度计算验证:")
    context_indices = torch.randint(0, V, (2, 3))
    target_indices = torch.randint(0, V, (2,))
    
    # 使用requires_grad
    W_grad = torch.randn(V, d, requires_grad=True)
    W_out_grad = torch.randn(d, V, requires_grad=True)
    
    loss_grad, _, _ = cbow_forward(context_indices, target_indices, W_grad, W_out_grad)
    loss_grad.backward()
    
    print(f"W梯度形状: {W_grad.grad.shape}")
    print(f"W_out梯度形状: {W_out_grad.grad.shape}")
    print("✓ 梯度计算成功")

测试数据:
上下文索引 (batch_size=3, context_size=2):
tensor([[2, 7],
        [6, 4],
        [6, 5]])

目标索引 (batch_size=3):
tensor([0, 4, 0])

输入权重 W 形状: torch.Size([10, 4])
输出权重 W_out 形状: torch.Size([4, 10])
--------------------------------------------------
CBOW前向传播结果:
损失值: 2.3026
隐藏层形状: torch.Size([3, 4])
隐藏层:
tensor([[ 0.0048,  0.0075, -0.0085,  0.0058],
        [-0.0067, -0.0021,  0.0038, -0.0080],
        [-0.0028, -0.0112,  0.0015, -0.0094]])

输出概率分布形状: torch.Size([3, 10])
输出概率分布 (前3个词):
tensor([[0.1000, 0.1000, 0.1000],
        [0.1000, 0.1000, 0.1000],
        [0.1000, 0.1000, 0.1000]])

验证: 每行概率和为 tensor([1.0000, 1.0000, 1.0000])

使用CBOW类测试:
损失值: 2.3026
隐藏层形状: torch.Size([3, 4])
概率分布形状: torch.Size([3, 10])

梯度计算验证:
W梯度形状: torch.Size([10, 4])
W_out梯度形状: torch.Size([4, 10])
✓ 梯度计算成功


## 6.1 理论计算题

给定查询矩阵 $Q \in \mathbb{R}^{2 \times 4}$，键矩阵 $K \in \mathbb{R}^{3 \times 4}$，值矩阵 $V \in \mathbb{R}^{3 \times 5}$。计算缩放点积注意力（无掩码）的输出矩阵。

### 给定矩阵

假设具体的数值矩阵为：

$$ Q = \begin{bmatrix} 1 & 0 & 2 & 1 \\ 0 & 1 & 1 & 2 \end{bmatrix} $$

$$ K = \begin{bmatrix} 1 & 2 & 0 & 1 \\ 0 & 1 & 1 & 0 \\ 2 & 0 & 1 & 1 \end{bmatrix} $$

$$ V = \begin{bmatrix} 1 & 0 & 2 & 1 & 0 \\ 0 & 1 & 1 & 2 & 1 \\ 2 & 0 & 1 & 0 & 1 \end{bmatrix} $$

其中 $d_k = 4$。

### 步骤1：计算得分矩阵 $S = QK^T / \sqrt{d_k}$

首先计算 $QK^T$：

$$ QK^T = \begin{bmatrix} 1 & 0 & 2 & 1 \\ 0 & 1 & 1 & 2 \end{bmatrix} \begin{bmatrix} 1 & 0 & 2 \\ 2 & 1 & 0 \\ 0 & 1 & 1 \\ 1 & 0 & 1 \end{bmatrix} $$

计算第一个元素 $(1,1)$：
$$ 1 \times 1 + 0 \times 2 + 2 \times 0 + 1 \times 1 = 1 + 0 + 0 + 1 = 2 $$

计算 $(1,2)$：
$$ 1 \times 0 + 0 \times 1 + 2 \times 1 + 1 \times 0 = 0 + 0 + 2 + 0 = 2 $$

计算 $(1,3)$：
$$ 1 \times 2 + 0 \times 0 + 2 \times 1 + 1 \times 1 = 2 + 0 + 2 + 1 = 5 $$

计算 $(2,1)$：
$$ 0 \times 1 + 1 \times 2 + 1 \times 0 + 2 \times 1 = 0 + 2 + 0 + 2 = 4 $$

计算 $(2,2)$：
$$ 0 \times 0 + 1 \times 1 + 1 \times 1 + 2 \times 0 = 0 + 1 + 1 + 0 = 2 $$

计算 $(2,3)$：
$$ 0 \times 2 + 1 \times 0 + 1 \times 1 + 2 \times 1 = 0 + 0 + 1 + 2 = 3 $$

因此：
$$ QK^T = \begin{bmatrix} 2 & 2 & 5 \\ 4 & 2 & 3 \end{bmatrix} $$

缩放因子 $\sqrt{d_k} = \sqrt{4} = 2$：

$$ S = \frac{1}{2} \begin{bmatrix} 2 & 2 & 5 \\ 4 & 2 & 3 \end{bmatrix} = \begin{bmatrix} 1 & 1 & 2.5 \\ 2 & 1 & 1.5 \end{bmatrix} $$

### 步骤2：对得分矩阵应用 softmax

对每一行应用 softmax：

**第一行**：$[1, 1, 2.5]$

计算指数：
$$ e^1 = 2.718, \quad e^1 = 2.718, \quad e^{2.5} = 12.182 $$

和：$2.718 + 2.718 + 12.182 = 17.618$

softmax值：
$$ p_{1,1} = \frac{2.718}{17.618} = 0.154 $$
$$ p_{1,2} = \frac{2.718}{17.618} = 0.154 $$
$$ p_{1,3} = \frac{12.182}{17.618} = 0.692 $$

**第二行**：$[2, 1, 1.5]$

计算指数：
$$ e^2 = 7.389, \quad e^1 = 2.718, \quad e^{1.5} = 4.482 $$

和：$7.389 + 2.718 + 4.482 = 14.589$

softmax值：
$$ p_{2,1} = \frac{7.389}{14.589} = 0.506 $$
$$ p_{2,2} = \frac{2.718}{14.589} = 0.186 $$
$$ p_{2,3} = \frac{4.482}{14.589} = 0.307 $$

因此注意力权重矩阵：
$$ P = \text{softmax}(S) = \begin{bmatrix} 0.154 & 0.154 & 0.692 \\ 0.506 & 0.186 & 0.307 \end{bmatrix} $$

### 步骤3：加权求和得到输出 $O = P \times V$

$$ O = \begin{bmatrix} 0.154 & 0.154 & 0.692 \\ 0.506 & 0.186 & 0.307 \end{bmatrix} \begin{bmatrix} 1 & 0 & 2 & 1 & 0 \\ 0 & 1 & 1 & 2 & 1 \\ 2 & 0 & 1 & 0 & 1 \end{bmatrix} $$

**第一行输出**：

第1列：$0.154 \times 1 + 0.154 \times 0 + 0.692 \times 2 = 0.154 + 0 + 1.384 = 1.538$

第2列：$0.154 \times 0 + 0.154 \times 1 + 0.692 \times 0 = 0 + 0.154 + 0 = 0.154$

第3列：$0.154 \times 2 + 0.154 \times 1 + 0.692 \times 1 = 0.308 + 0.154 + 0.692 = 1.154$

第4列：$0.154 \times 1 + 0.154 \times 2 + 0.692 \times 0 = 0.154 + 0.308 + 0 = 0.462$

第5列：$0.154 \times 0 + 0.154 \times 1 + 0.692 \times 1 = 0 + 0.154 + 0.692 = 0.846$

**第二行输出**：

第1列：$0.506 \times 1 + 0.186 \times 0 + 0.307 \times 2 = 0.506 + 0 + 0.614 = 1.120$

第2列：$0.506 \times 0 + 0.186 \times 1 + 0.307 \times 0 = 0 + 0.186 + 0 = 0.186$

第3列：$0.506 \times 2 + 0.186 \times 1 + 0.307 \times 1 = 1.012 + 0.186 + 0.307 = 1.505$

第4列：$0.506 \times 1 + 0.186 \times 2 + 0.307 \times 0 = 0.506 + 0.372 + 0 = 0.878$

第5列：$0.506 \times 0 + 0.186 \times 1 + 0.307 \times 1 = 0 + 0.186 + 0.307 = 0.493$

### 最终输出矩阵

$$ \boxed{O = \begin{bmatrix} 1.538 & 0.154 & 1.154 & 0.462 & 0.846 \\ 1.120 & 0.186 & 1.505 & 0.878 & 0.493 \end{bmatrix}} $$

### 中间步骤总结

1. **得分矩阵** $S = QK^T/\sqrt{d_k}$：
   $$ S = \begin{bmatrix} 1 & 1 & 2.5 \\ 2 & 1 & 1.5 \end{bmatrix} $$

2. **注意力权重** $P = \text{softmax}(S)$：
   $$ P = \begin{bmatrix} 0.154 & 0.154 & 0.692 \\ 0.506 & 0.186 & 0.307 \end{bmatrix} $$

3. **输出矩阵** $O = P \times V$：
   $$ O = \begin{bmatrix} 1.538 & 0.154 & 1.154 & 0.462 & 0.846 \\ 1.120 & 0.186 & 1.505 & 0.878 & 0.493 \end{bmatrix} $$

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        """
        多头注意力机制
        
        参数:
            d_model: 模型维度
            num_heads: 注意力头数
        """
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model必须能被num_heads整除"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.d_v = d_model // num_heads
        
        # 线性投影层
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
    def forward(self, X):
        """
        前向传播
        
        参数:
            X: 输入序列，形状 (seq_len, batch, d_model)
        
        返回:
            output: 输出序列，形状 (seq_len, batch, d_model)
        """
        seq_len, batch_size, _ = X.shape
        
        # 1. 线性投影得到Q, K, V
        Q = self.W_q(X)  # (seq_len, batch, d_model)
        K = self.W_k(X)  # (seq_len, batch, d_model)
        V = self.W_v(X)  # (seq_len, batch, d_model)
        
        # 2. 重塑为多头形式
        # (seq_len, batch, d_model) -> (seq_len, batch, num_heads, d_k)
        Q = Q.view(seq_len, batch_size, self.num_heads, self.d_k)
        K = K.view(seq_len, batch_size, self.num_heads, self.d_k)
        V = V.view(seq_len, batch_size, self.num_heads, self.d_v)
        
        # 转置为 (batch, num_heads, seq_len, d_k)
        Q = Q.transpose(0, 2).transpose(1, 0)  # (batch, num_heads, seq_len, d_k)
        K = K.transpose(0, 2).transpose(1, 0)  # (batch, num_heads, seq_len, d_k)
        V = V.transpose(0, 2).transpose(1, 0)  # (batch, num_heads, seq_len, d_v)
        
        # 3. 计算缩放点积注意力
        # 得分矩阵: Q @ K^T / sqrt(d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)  # (batch, num_heads, seq_len, seq_len)
        
        # softmax
        attention_weights = F.softmax(scores, dim=-1)  # (batch, num_heads, seq_len, seq_len)
        
        # 加权求和
        attention_output = torch.matmul(attention_weights, V)  # (batch, num_heads, seq_len, d_v)
        
        # 4. 合并多头
        # (batch, num_heads, seq_len, d_v) -> (batch, seq_len, num_heads, d_v)
        attention_output = attention_output.transpose(1, 0).transpose(0, 1)  # (seq_len, batch, num_heads, d_v)
        
        # 合并为 (seq_len, batch, d_model)
        concat_output = attention_output.contiguous().view(seq_len, batch_size, self.d_model)
        
        # 5. 最终线性层
        output = self.W_o(concat_output)  # (seq_len, batch, d_model)
        
        return output


# 手动实现的多头注意力（不依赖nn.Linear，展示详细计算过程）
class MultiHeadAttentionManual(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttentionManual, self).__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.d_v = d_model // num_heads
        
        # 初始化权重矩阵
        self.W_q = torch.randn(d_model, d_model) * 0.01
        self.W_k = torch.randn(d_model, d_model) * 0.01
        self.W_v = torch.randn(d_model, d_model) * 0.01
        self.W_o = torch.randn(d_model, d_model) * 0.01
        
        # 偏置（可选，这里简单处理）
        self.b_q = torch.zeros(d_model)
        self.b_k = torch.zeros(d_model)
        self.b_v = torch.zeros(d_model)
        self.b_o = torch.zeros(d_model)
    
    def forward(self, X):
        seq_len, batch_size, _ = X.shape
        
        # 1. 线性投影
        Q = torch.matmul(X, self.W_q) + self.b_q  # (seq_len, batch, d_model)
        K = torch.matmul(X, self.W_k) + self.b_k
        V = torch.matmul(X, self.W_v) + self.b_v
        
        # 2. 重塑为多头
        Q = Q.view(seq_len, batch_size, self.num_heads, self.d_k)
        K = K.view(seq_len, batch_size, self.num_heads, self.d_k)
        V = V.view(seq_len, batch_size, self.num_heads, self.d_v)
        
        # 转置
        Q = Q.permute(1, 2, 0, 3)  # (batch, num_heads, seq_len, d_k)
        K = K.permute(1, 2, 0, 3)
        V = V.permute(1, 2, 0, 3)
        
        # 3. 缩放点积注意力
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attention_weights = F.softmax(scores, dim=-1)
        attention_output = torch.matmul(attention_weights, V)
        
        # 4. 合并多头
        attention_output = attention_output.permute(2, 0, 1, 3)  # (seq_len, batch, num_heads, d_v)
        concat_output = attention_output.contiguous().view(seq_len, batch_size, self.d_model)
        
        # 5. 最终线性层
        output = torch.matmul(concat_output, self.W_o) + self.b_o
        
        return output


# 测试代码
if __name__ == "__main__":
    # 设置参数
    d_model = 4
    num_heads = 2
    seq_len = 3
    batch_size = 2
    
    print("参数设置:")
    print(f"d_model = {d_model}")
    print(f"num_heads = {num_heads}")
    print(f"每个头的维度 d_k = d_v = {d_model // num_heads}")
    print(f"序列长度 = {seq_len}")
    print(f"批次大小 = {batch_size}")
    print("-" * 50)
    
    # 创建随机输入
    torch.manual_seed(42)
    X = torch.randn(seq_len, batch_size, d_model)
    
    print(f"输入 X 形状: {X.shape}")
    print(f"输入 X:\n{X}")
    print("-" * 50)
    
    # 测试PyTorch实现
    model = MultiHeadAttention(d_model, num_heads)
    output = model(X)
    
    print("PyTorch实现:")
    print(f"输出形状: {output.shape}")  # 期望: (seq_len, batch, d_model)
    print(f"输出:\n{output}")
    print("-" * 50)
    
    # 测试手动实现
    model_manual = MultiHeadAttentionManual(d_model, num_heads)
    output_manual = model_manual(X)
    
    print("手动实现:")
    print(f"输出形状: {output_manual.shape}")
    print(f"输出:\n{output_manual}")
    print("-" * 50)
    
    # 验证形状
    assert output.shape == (seq_len, batch_size, d_model), "输出形状错误"
    assert output_manual.shape == (seq_len, batch_size, d_model), "手动实现输出形状错误"
    print("✓ 所有形状验证通过")
    
    # 详细展示多头注意力的中间步骤（使用PyTorch实现）
    print("\n" + "=" * 50)
    print("详细中间步骤（使用PyTorch实现）:")
    
    # 重新运行并提取中间结果
    with torch.no_grad():
        # 获取投影后的Q, K, V
        Q_proj = model.W_q(X)
        K_proj = model.W_k(X)
        V_proj = model.W_v(X)
        
        print(f"投影后 Q 形状: {Q_proj.shape}")
        print(f"投影后 K 形状: {K_proj.shape}")
        print(f"投影后 V 形状: {V_proj.shape}")
        
        # 重塑为多头
        Q_multi = Q_proj.view(seq_len, batch_size, num_heads, d_model//num_heads)
        Q_multi = Q_multi.transpose(0, 2).transpose(1, 0)
        print(f"多头 Q 形状: {Q_multi.shape}")  # (batch, num_heads, seq_len, d_k)
        
        # 计算注意力权重（只展示第一个头）
        scores = torch.matmul(Q_multi, Q_multi.transpose(-2, -1)) / math.sqrt(d_model//num_heads)
        weights = F.softmax(scores, dim=-1)
        print(f"注意力权重形状: {weights.shape}")  # (batch, num_heads, seq_len, seq_len)
        print(f"第一个头的注意力权重（第1个batch）:\n{weights[0, 0]}")

参数设置:
d_model = 4
num_heads = 2
每个头的维度 d_k = d_v = 2
序列长度 = 3
批次大小 = 2
--------------------------------------------------
输入 X 形状: torch.Size([3, 2, 4])
输入 X:
tensor([[[ 1.9269,  1.4873,  0.9007, -2.1055],
         [ 0.6784, -1.2345, -0.0431, -1.6047]],

        [[ 0.3559, -0.6866, -0.4934,  0.2415],
         [-1.1109,  0.0915, -2.3169, -0.2168]],

        [[-0.3097, -0.3957,  0.8034, -0.6216],
         [-0.5920, -0.0631, -0.8286,  0.3309]]])
--------------------------------------------------
PyTorch实现:
输出形状: torch.Size([3, 2, 4])
输出:
tensor([[[-0.1992, -0.9688, -0.2928, -0.7249],
         [ 0.1418, -0.4346, -0.7022, -0.3377]],

        [[ 0.1621, -0.1296, -0.1053,  0.0097],
         [ 0.1553, -0.2454, -0.2753, -0.0504]],

        [[ 0.0242, -0.1079, -0.3783,  0.1012],
         [ 0.2049, -0.1977, -0.1621, -0.0988]]], grad_fn=<ViewBackward0>)
--------------------------------------------------
手动实现:
输出形状: torch.Size([3, 2, 4])
输出:
tensor([[[-6.6705e-05,  6.4974e-05,  2.9992e-04,  2.4900e